# Smart MCQ Solver

Ankur · 21f2000153 · Deep Learning & GenAI (T2 2026)

Predict the top three answers out of five options (A-E). Metric: MAP@3.

Three models:

1. Logistic regression
2. Self-attention model built from scratch
3. Qwen2.5-32B-Instruct (pretrained, zero-shot)

Set `RUN_QWEN = False` to run models 1 and 2 only.

## 1. Setup

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate wandb 2>&1 | tail -2

import bitsandbytes
print("bitsandbytes:", bitsandbytes.__version__)

In [ ]:
import re
import math
import itertools

import numpy as np
import pandas as pd

RUN_QWEN = True
USE_WANDB = True

WANDB_PROJECT = "smart-mcq-solver"
WANDB_ENTITY  = "21f2000153-dl-genai-project"

SEED = 42
VALIDATION_FRACTION = 0.20
QWEN_VALIDATION_ROWS = 150
NUMBER_OF_PERMUTATIONS = 10

DATA_FOLDER = "/kaggle/input/competitions/smart-mcq-solver-challenge/"

OPTIONS = ["A", "B", "C", "D", "E"]
LETTER_TO_INDEX = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

np.random.seed(SEED)

In [ ]:
train_df = pd.read_csv(DATA_FOLDER + "train.csv")
test_df = pd.read_csv(DATA_FOLDER + "test.csv")
sample_submission = pd.read_csv(DATA_FOLDER + "sample_submission.csv")

print("train rows:", len(train_df))
print("test rows :", len(test_df))
print("submission columns:", list(sample_submission.columns))

train_df.head(2)

## 2. Data

### 2.1 Remove the templated filler from the prompts

In [ ]:
PREFIXES = [
    "Pick the best possible answer:",
    "Select the most accurate option:",
    "Identify the correct statement:",
    "Determine the correct option:",
    "Choose the correct answer:",
    "Which of the following is correct?",
]

SUFFIXES = [
    "among the listed options.",
    "from the following choices.",
    "carefully.",
    "based on the given context.",
    "among the list of options.",
]

def clean_prompt(text):
    text = str(text).strip()

    for prefix in PREFIXES:
        if text.startswith(prefix):
            text = text[len(prefix):]
            text = text.strip()
            break

    for suffix in SUFFIXES:
        if text.endswith(suffix):
            text = text[:-len(suffix)]
            text = text.strip()
            break

    return text

train_df["question"] = train_df["prompt"].apply(clean_prompt)
test_df["question"] = test_df["prompt"].apply(clean_prompt)

number_changed = (train_df["question"] != train_df["prompt"].str.strip()).sum()
print("filler removed from", number_changed, "of", len(train_df), "rows")
print()
print("before:", train_df["prompt"].iloc[0][:90])
print("after :", train_df["question"].iloc[0][:90])

y_all = train_df["answer"].map(LETTER_TO_INDEX).values

### 2.2 Basic EDA

In [ ]:
import matplotlib.pyplot as plt

print("train rows :", len(train_df))
print("test rows  :", len(test_df))
print("missing values in train:", train_df.isna().sum().sum())
print("missing values in test :", test_df.isna().sum().sum())

print()
print("how often each letter is the answer:")
answer_counts = train_df["answer"].value_counts().reindex(OPTIONS)
print(answer_counts.to_string())

In [ ]:
question_lengths = train_df["question"].astype(str).str.len()

option_lengths = []
for letter in OPTIONS:
    option_lengths.append(train_df[letter].astype(str).str.len())
option_lengths = np.column_stack(option_lengths)

print("question length in characters: mean", int(question_lengths.mean()),
      " min", question_lengths.min(), " max", question_lengths.max())
print("option length in characters  : mean", int(option_lengths.mean()),
      " min", option_lengths.min(), " max", option_lengths.max())

In [ ]:
correct_lengths = []
wrong_lengths = []

for row_number in range(len(train_df)):
    correct_option = y_all[row_number]
    for option_number in range(5):
        if option_number == correct_option:
            correct_lengths.append(option_lengths[row_number, option_number])
        else:
            wrong_lengths.append(option_lengths[row_number, option_number])

print("average length of the CORRECT option:", int(np.mean(correct_lengths)))
print("average length of the WRONG options :", int(np.mean(wrong_lengths)))

longest = option_lengths.argmax(axis=1)
print()
print("the answer is the longest option in",
      round((longest == y_all).mean() * 100, 1), "% of questions")

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(answer_counts.index, answer_counts.values)
axes[0].set_title("How often each letter is the answer")

axes[1].hist(question_lengths, bins=30)
axes[1].set_title("Question length (characters)")

axes[2].hist(correct_lengths, bins=30, alpha=0.6, label="correct")
axes[2].hist(wrong_lengths, bins=30, alpha=0.6, label="wrong")
axes[2].set_title("Option length: correct vs wrong")
axes[2].legend()

plt.tight_layout()
plt.show()

The correct option is on average longer than the wrong ones. This is because
the wrong options were made by rewriting the correct answer. It is a property of
this dataset, not of multiple choice questions in general.

### 2.3 Find duplicate questions

In [ ]:
def normalize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = text.strip()
    return text

def make_question_key(row):
    option_texts = []
    for letter in OPTIONS:
        option_texts.append(normalize(row[letter]))
    option_texts.sort()
    return "|".join(option_texts)

train_df["key"] = train_df.apply(make_question_key, axis=1)

key_counts = train_df["key"].value_counts()
number_repeated = (key_counts > 1).sum()
rows_in_repeats = key_counts[key_counts > 1].sum()

print("total rows        :", len(train_df))
print("unique questions  :", train_df["key"].nunique())
print("questions repeated:", number_repeated)
print("rows in repeats   :", rows_in_repeats)

### 2.4 Grouped train/validation split

In [ ]:
unique_keys = train_df["key"].unique().tolist()

shuffler = np.random.RandomState(SEED)
shuffler.shuffle(unique_keys)

number_of_validation_keys = int(VALIDATION_FRACTION * len(unique_keys))
validation_keys = set(unique_keys[:number_of_validation_keys])

is_validation = train_df["key"].isin(validation_keys).values

val_df = train_df[is_validation].reset_index(drop=True)
fit_df = train_df[~is_validation].reset_index(drop=True)

y_val = val_df["answer"].map(LETTER_TO_INDEX).values
y_fit = fit_df["answer"].map(LETTER_TO_INDEX).values

shared = set(val_df["key"]) & set(fit_df["key"])

print("rows used for fitting :", len(fit_df))
print("rows used for testing :", len(val_df))
print("questions on both sides:", len(shared), " (must be 0)")

### 2.5 MAP@3 metric

In [ ]:
def map3(scores, labels):
    total = 0.0

    for i in range(len(labels)):
        current_row = scores[i]

        ranking = sorted(range(5), key=lambda j: current_row[j], reverse=True)

        position = ranking.index(labels[i])

        if position < 3:
            total = total + 1 / (position + 1)

    return total / len(labels)

def accuracy(scores, labels):
    best_option = scores.argmax(axis=1)
    return float((best_option == labels).mean())

def macro_f1(scores, labels):
    from sklearn.metrics import f1_score
    best_option = scores.argmax(axis=1)
    return float(f1_score(labels, best_option, average="macro",
                          labels=[0, 1, 2, 3, 4], zero_division=0))

def cross_entropy_loss(scores, labels):
    scores = np.asarray(scores, dtype=float)
    shifted = scores - scores.max(axis=1, keepdims=True)
    exponentials = np.exp(shifted)
    probabilities = exponentials / exponentials.sum(axis=1, keepdims=True)

    total = 0.0
    for i in range(len(labels)):
        probability_of_correct = probabilities[i, labels[i]]
        total = total - np.log(probability_of_correct + 1e-12)

    return float(total / len(labels))


def evaluate(scores, labels, name):
    results = {
        "val_accuracy": accuracy(scores, labels),
        "val_macro_f1": macro_f1(scores, labels),
        "val_map3": float(map3(scores, labels)),
        "val_loss": cross_entropy_loss(scores, labels),
    }
    print(name)
    print("   accuracy :", round(results["val_accuracy"], 4))
    print("   macro F1 :", round(results["val_macro_f1"], 4))
    print("   MAP@3    :", round(results["val_map3"], 4))
    print("   val loss :", round(results["val_loss"], 4))
    return results

In [ ]:
perfect_scores = np.array([[9, 0, 0, 0, 0], [0, 9, 0, 0, 0]])
second_place = np.array([[0, 9, 0, 0, 0], [9, 0, 0, 0, 0]])
labels = np.array([0, 1])

print("answer ranked first  :", map3(perfect_scores, labels), "(expect 1.0)")
print("answer ranked second :", map3(second_place, labels), "(expect 0.5)")
print("uniform random guess :", round((1 + 1/2 + 1/3) / 5, 4))

### 2.6 Length baseline

In [ ]:
def length_scores(dataframe):
    columns = []
    for letter in OPTIONS:
        lengths = dataframe[letter].astype(str).str.len()
        columns.append(lengths)
    return np.column_stack(columns).astype(float)

baseline_scores = length_scores(val_df)
baseline_results = evaluate(baseline_scores, y_val, "LENGTH BASELINE")

### 2.7 Weights & Biases setup

In [ ]:
if USE_WANDB:
    import wandb
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    wandb.login(key=secrets.get_secret("WANDB_API_KEY"))
    print("logged in to Weights & Biases")

def log_to_wandb(run_name, settings, results, per_epoch=None):
    if not USE_WANDB:
        return

    import wandb

    wandb.init(project=WANDB_PROJECT, name=run_name, config=settings, reinit=True)

    if per_epoch is not None:
        for one_epoch in per_epoch:
            wandb.log(one_epoch)

    wandb.log(results)
    for key in results:
        wandb.summary[key] = results[key]

    wandb.finish()
    print("logged run:", run_name)

## 3. Model 1 — Logistic regression

Two features per option: character length and word count.

### Features

In [ ]:
from sklearn.preprocessing import StandardScaler

def count_characters(dataframe):
    columns = []
    for letter in OPTIONS:
        columns.append(dataframe[letter].astype(str).str.len())
    return np.column_stack(columns).astype(float)

def count_words(dataframe):
    columns = []
    for letter in OPTIONS:
        columns.append(dataframe[letter].astype(str).str.split().str.len())
    return np.column_stack(columns).astype(float)

In [ ]:
FEATURE_NAMES = ["length", "wordcount"]

def build_features(dataframe):
    feature_1 = count_characters(dataframe)
    feature_2 = count_words(dataframe)
    return np.stack([feature_1, feature_2], axis=-1)

features_fit = build_features(fit_df)
features_val = build_features(val_df)

print("feature array shape:", features_fit.shape)
print("  = (questions, options, features)")

### How good is each feature alone?

In [ ]:
for feature_number in range(2):
    one_feature = features_fit[:, :, feature_number]
    picked = one_feature.argmax(axis=1)
    score = (picked == y_fit).mean()

    name = FEATURE_NAMES[feature_number]
    bar = "#" * int(score * 50)
    print(f"  {name:<18} {score:.4f}   {bar}")

print(f"  {'random guessing':<18} 0.2000")

In [ ]:
from sklearn.linear_model import LogisticRegression

n_questions = len(fit_df)

training_rows = features_fit.reshape(n_questions * 5, 2)

training_labels = np.zeros((n_questions, 5))
for question_number in range(n_questions):
    correct_option = y_fit[question_number]
    training_labels[question_number, correct_option] = 1
training_labels = training_labels.reshape(n_questions * 5)

scaler = StandardScaler()
training_rows = scaler.fit_transform(training_rows)

ranker = LogisticRegression(max_iter=1000)
ranker.fit(training_rows, training_labels)

def ranker_scores(features):
    n = features.shape[0]
    flat = features.reshape(n * 5, 2)
    flat = scaler.transform(flat)
    probabilities = ranker.predict_proba(flat)[:, 1]
    return probabilities.reshape(n, 5)

print("learned weights:")
for feature_number in range(2):
    name = FEATURE_NAMES[feature_number]
    weight = ranker.coef_[0][feature_number]
    print(f"  {name:<18} {weight:+.4f}")

In [ ]:
ranker_train_scores = ranker_scores(features_fit)
ranker_val_scores = ranker_scores(features_val)

ranker_train_loss = cross_entropy_loss(ranker_train_scores, y_fit)
ranker_results = evaluate(ranker_val_scores, y_val, "MODEL 1 - logistic regression")
ranker_results["train_loss"] = ranker_train_loss
print("   train loss:", round(ranker_train_loss, 4))

log_to_wandb(
    "model1-feature-ranker",
    {
        "model": "logistic_regression",
        "features": FEATURE_NAMES,
        "trainable_params": 2,
        "split": "grouped",
        "validation_rows": len(val_df),
    },
    ranker_results,
)

### Error analysis

In [ ]:
longest_option = length_scores(val_df).argmax(axis=1)
answer_is_longest = (longest_option == y_val)

ranker_picked = ranker_val_scores.argmax(axis=1)
ranker_was_right = (ranker_picked == y_val)

group_1 = ranker_was_right[answer_is_longest]
group_2 = ranker_was_right[~answer_is_longest]

print("when the answer IS the longest option:")
print("   questions:", len(group_1), "  accuracy:", round(group_1.mean(), 4))
print()
print("when the answer is NOT the longest option:")
print("   questions:", len(group_2), "  accuracy:", round(group_2.mean(), 4))

The model mostly picks the longest option.

## 4. Model 2 — Self-attention from scratch

Embedding, self-attention, mean pooling, one linear layer. Built without `nn.Transformer`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter

PAD_ID = 0
UNK_ID = 1

MAX_LENGTH = 32
EMBEDDING_DIM = 32
EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 0.001

torch.manual_seed(SEED)

### Tokenizer and vocabulary

In [ ]:
def tokenize(text):
    text = str(text).lower()
    return re.findall(r"[a-z0-9]+", text)

texts_for_vocabulary = []
for question in fit_df["question"]:
    texts_for_vocabulary.append(question)
for row_number in range(len(fit_df)):
    row = fit_df.iloc[row_number]
    for letter in OPTIONS:
        texts_for_vocabulary.append(row[letter])

word_counts = Counter()
for text in texts_for_vocabulary:
    for word in tokenize(text):
        word_counts[word] += 1

vocabulary = {"<pad>": PAD_ID, "<unk>": UNK_ID}
for word in word_counts:
    vocabulary[word] = len(vocabulary)

print("distinct words seen :", len(word_counts))
print("words kept in vocab :", len(vocabulary))

In [ ]:
def encode_text(text):
    word_ids = []
    for word in tokenize(text):
        if word in vocabulary:
            word_ids.append(vocabulary[word])
        else:
            word_ids.append(UNK_ID)

    word_ids = word_ids[:MAX_LENGTH]
    while len(word_ids) < MAX_LENGTH:
        word_ids.append(PAD_ID)

    return word_ids

def dataframe_to_tensors(dataframe):
    question_ids = []
    for question in dataframe["question"]:
        question_ids.append(encode_text(question))

    option_ids = []
    for row_number in range(len(dataframe)):
        row = dataframe.iloc[row_number]
        this_row = []
        for letter in OPTIONS:
            this_row.append(encode_text(row[letter]))
        option_ids.append(this_row)

    return torch.tensor(question_ids), torch.tensor(option_ids)

questions_fit, options_fit = dataframe_to_tensors(fit_df)
questions_val, options_val = dataframe_to_tensors(val_df)

print("question tensor:", tuple(questions_fit.shape))
print("option tensor  :", tuple(options_fit.shape), " = (rows, 5 options, words)")

### Self-attention

In [ ]:
class SelfAttention(nn.Module):

    def __init__(self, dimension):
        super().__init__()
        self.make_query = nn.Linear(dimension, dimension)
        self.make_key = nn.Linear(dimension, dimension)
        self.make_value = nn.Linear(dimension, dimension)
        self.dimension = dimension

    def forward(self, x, real_word_mask):
        query = self.make_query(x)
        key = self.make_key(x)
        value = self.make_value(x)

        scores = query @ key.transpose(1, 2)
        scores = scores / math.sqrt(self.dimension)

        padding_mask = ~real_word_mask.unsqueeze(1)
        scores = scores.masked_fill(padding_mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)
        return weights @ value

### The model

In [ ]:
class ScratchAttentionModel(nn.Module):

    def __init__(self, vocabulary_size, dimension=EMBEDDING_DIM):
        super().__init__()
        self.word_embedding = nn.Embedding(vocabulary_size, dimension,
                                           padding_idx=PAD_ID)
        self.attention = SelfAttention(dimension)

        self.scorer = nn.Linear(dimension * 2, 1)

    def encode_sentence(self, word_ids):
        real_word_mask = (word_ids != PAD_ID)

        x = self.word_embedding(word_ids)
        x = self.attention(x, real_word_mask)

        mask = real_word_mask.unsqueeze(-1).float()
        total = (x * mask).sum(dim=1)
        count = mask.sum(dim=1).clamp(min=1)
        return total / count

    def forward(self, question_ids, option_ids):
        batch_size = option_ids.shape[0]
        number_of_options = option_ids.shape[1]
        sentence_length = option_ids.shape[2]

        question_vector = self.encode_sentence(question_ids)

        flat_options = option_ids.reshape(batch_size * number_of_options,
                                          sentence_length)
        option_vectors = self.encode_sentence(flat_options)

        repeated = question_vector.unsqueeze(1)
        repeated = repeated.expand(batch_size, number_of_options, -1)
        repeated = repeated.reshape(batch_size * number_of_options, -1)

        combined = torch.cat([repeated, option_vectors], dim=1)

        scores = self.scorer(combined)
        return scores.reshape(batch_size, number_of_options)

device = "cuda" if torch.cuda.is_available() else "cpu"
attention_model = ScratchAttentionModel(len(vocabulary)).to(device)

number_of_parameters = sum(p.numel() for p in attention_model.parameters())
print("parameters:", format(number_of_parameters, ","))
print("device    :", device)

### Training

In [ ]:
optimizer = torch.optim.Adam(attention_model.parameters(), lr=LEARNING_RATE)
loss_function = nn.CrossEntropyLoss()

labels_fit = torch.tensor(y_fit)
labels_val = torch.tensor(y_val)

epoch_history = []
best_map3 = 0.0
best_val_scores = None

for epoch in range(1, EPOCHS + 1):

    attention_model.train()
    shuffled_order = torch.randperm(len(questions_fit))
    losses_this_epoch = []

    for start in range(0, len(shuffled_order), BATCH_SIZE):
        batch = shuffled_order[start:start + BATCH_SIZE]

        optimizer.zero_grad()

        predictions = attention_model(questions_fit[batch].to(device),
                                  options_fit[batch].to(device))
        loss = loss_function(predictions, labels_fit[batch].to(device))

        loss.backward()
        optimizer.step()

        losses_this_epoch.append(loss.item())

    attention_model.eval()
    with torch.no_grad():
        val_outputs = attention_model(questions_val.to(device),
                                      options_val.to(device))
        val_loss = loss_function(val_outputs, labels_val.to(device)).item()
        val_scores = val_outputs.cpu().numpy()

    average_loss = float(np.mean(losses_this_epoch))
    this_accuracy = accuracy(val_scores, y_val)
    this_map3 = float(map3(val_scores, y_val))

    if this_map3 > best_map3:
        best_map3 = this_map3
        best_val_scores = val_scores

    epoch_history.append({
        "epoch": epoch,
        "train_loss": average_loss,
        "val_loss": float(val_loss),
        "val_accuracy": this_accuracy,
        "val_map3": this_map3,
    })

    print(f"epoch {epoch:>2}   train_loss {average_loss:.4f}   "
          f"val_loss {val_loss:.4f}   "
          f"accuracy {this_accuracy:.4f}   MAP@3 {this_map3:.4f}")

In [ ]:
transformer_results = evaluate(best_val_scores, y_val, "MODEL 2 - scratch attention")

log_to_wandb(
    "model2-scratch-transformer",
    {
        "model": "scratch_transformer",
        "dimension": EMBEDDING_DIM,
        "layers": 1,
        "attention_heads": 1,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "optimizer": "Adam",
        "trainable_params": number_of_parameters,
        "vocabulary_size": len(vocabulary),
        "split": "grouped",
        "validation_rows": len(val_df),
    },
    transformer_results,
    epoch_history,
)

Train loss keeps falling while validation loss rises after about epoch 2. That gap is overfitting: the model is memorising the training rows. The epoch with the best validation MAP@3 is the one kept.

## 5. Model 3 — Qwen2.5-32B-Instruct

Pretrained model, zero-shot, loaded in 4-bit.

In [ ]:
if RUN_QWEN:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from tqdm.auto import tqdm

    MODEL_NAME = "Qwen/Qwen2.5-32B-Instruct"

    quantization_settings = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    language_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_settings,
        device_map="auto",
    )
    language_model.eval()

    print("model loaded")

In [ ]:
if RUN_QWEN:
    letter_token_ids = []
    for letter in OPTIONS:
        ids = tokenizer.encode(letter, add_special_tokens=False)
        print("letter", letter, "-> token ids", ids)
        letter_token_ids.append(ids[0])

### Averaging over option orderings

In [ ]:
if RUN_QWEN:
    all_possible_orderings = list(itertools.permutations(range(5)))
    np.random.RandomState(0).shuffle(all_possible_orderings)

    ORDERINGS = []
    for ordering in all_possible_orderings[:NUMBER_OF_PERMUTATIONS]:
        ORDERINGS.append(list(ordering))

    ORDERINGS_PER_BATCH = 5

    print("using", len(ORDERINGS), "orderings out of 120 possible")
    print("first ordering:", ORDERINGS[0], " (display slot -> original option)")

In [ ]:
if RUN_QWEN:

    def build_prompt(row, ordering):
        lines = []
        lines.append("Question: " + str(row["question"]))
        lines.append("")

        for slot in range(5):
            original_option = ordering[slot]
            option_text = str(row[OPTIONS[original_option]]).strip()
            lines.append(OPTIONS[slot] + ". " + option_text)

        lines.append("")
        lines.append("Answer with the single letter of the correct option.")

        text = "\n".join(lines)
        messages = [{"role": "user", "content": text}]
        return tokenizer.apply_chat_template(messages, tokenize=False,
                                             add_generation_prompt=True)

    def score_one_question(row):
        totals = np.zeros(5)

        for start in range(0, len(ORDERINGS), ORDERINGS_PER_BATCH):
            batch_of_orderings = ORDERINGS[start:start + ORDERINGS_PER_BATCH]

            prompts = []
            for ordering in batch_of_orderings:
                prompts.append(build_prompt(row, ordering))

            inputs = tokenizer(prompts, return_tensors="pt", padding=True)
            inputs = inputs.to(language_model.device)

            with torch.no_grad():
                output = language_model(**inputs)

            last_position = output.logits[:, -1, :]
            letter_logits = last_position[:, letter_token_ids]
            log_probabilities = torch.log_softmax(letter_logits.float(), dim=-1)
            log_probabilities = log_probabilities.cpu().numpy()

            for prompt_number in range(len(batch_of_orderings)):
                ordering = batch_of_orderings[prompt_number]
                for slot in range(5):
                    original_option = ordering[slot]
                    totals[original_option] += log_probabilities[prompt_number, slot]

            del inputs, output, last_position, letter_logits
            torch.cuda.empty_cache()

        return totals / len(ORDERINGS)

    def score_many_questions(dataframe):
        all_scores = []
        for row_number in tqdm(range(len(dataframe))):
            all_scores.append(score_one_question(dataframe.iloc[row_number]))
        return np.vstack(all_scores)

In [ ]:
if RUN_QWEN:
    for row_number in range(2):
        row = val_df.iloc[row_number]
        scores = score_one_question(row)

        print("Q:", row["question"][:70])
        for option_number in range(5):
            print("   ", OPTIONS[option_number], round(float(scores[option_number]), 3))
        print("   picked:", OPTIONS[int(scores.argmax())], "  correct:", row["answer"])
        print()

### Validation

In [ ]:
qwen_results = None
qwen_row_numbers = None

if RUN_QWEN:
    chooser = np.random.RandomState(SEED)
    qwen_row_numbers = chooser.choice(len(val_df),
                                      size=min(QWEN_VALIDATION_ROWS, len(val_df)),
                                      replace=False)

    qwen_val_df = val_df.iloc[qwen_row_numbers].reset_index(drop=True)
    y_qwen = qwen_val_df["answer"].map(LETTER_TO_INDEX).values

    qwen_val_scores = score_many_questions(qwen_val_df)

    qwen_results = evaluate(qwen_val_scores, y_qwen, "MODEL 3 - Qwen2.5-32B")

    log_to_wandb(
        "model3-qwen32b-zeroshot",
        {
            "model": MODEL_NAME,
            "method": "zero-shot letter log-probabilities",
            "quantization": "4-bit nf4",
            "orderings_averaged": NUMBER_OF_PERMUTATIONS,
            "trainable_params": 0,
            "split": "grouped",
            "validation_rows": len(qwen_val_df),
        },
        qwen_results,
    )

### Predicted letter distribution

In [ ]:
if RUN_QWEN:
    predicted_letters = []
    for option_number in qwen_val_scores.argmax(axis=1):
        predicted_letters.append(OPTIONS[option_number])

    true_counts = pd.Series(qwen_val_df["answer"]).value_counts()
    true_counts = true_counts.reindex(OPTIONS, fill_value=0)

    predicted_counts = pd.Series(predicted_letters).value_counts()
    predicted_counts = predicted_counts.reindex(OPTIONS, fill_value=0)

    comparison = pd.DataFrame({
        "true": true_counts,
        "predicted": predicted_counts,
    })
    comparison["difference"] = comparison["predicted"] - comparison["true"]

    print(comparison)
    print()
    print("total absolute imbalance:", int(comparison["difference"].abs().sum()))

## 6. Comparison

In [ ]:
if RUN_QWEN and qwen_results is not None:
    rows_used = qwen_row_numbers
    labels_used = y_val[rows_used]

    baseline_row = evaluate(baseline_scores[rows_used], labels_used,
                            "length baseline (same rows)")
    ranker_row = evaluate(ranker_val_scores[rows_used], labels_used,
                          "logistic regression (same rows)")
    transformer_row = evaluate(best_val_scores[rows_used], labels_used,
                               "scratch attention (same rows)")
    qwen_row = qwen_results
    comparison_note = f"all scored on the same {len(rows_used)} validation rows"
else:
    baseline_row = baseline_results
    ranker_row = ranker_results
    transformer_row = transformer_results
    qwen_row = None
    comparison_note = f"scored on all {len(val_df)} validation rows"

In [ ]:
table_rows = [
    ("Length baseline", "heuristic", 0, baseline_row),
    ("Model 1: logistic regression", "classical ML", 2, ranker_row),
    ("Model 2: scratch attention", "neural, from scratch",
     number_of_parameters, transformer_row),
]

if qwen_row is not None:
    table_rows.append(
        ("Model 3: Qwen2.5-32B", "pretrained LLM, zero-shot", 0, qwen_row))

summary = pd.DataFrame([
    {
        "Model": name,
        "Type": kind,
        "Trainable params": format(params, ","),
        "Accuracy": round(results["val_accuracy"], 4),
        "Macro F1": round(results["val_macro_f1"], 4),
        "MAP@3": round(results["val_map3"], 4),
    }
    for name, kind, params, results in table_rows
])

print(comparison_note)
print()
print(summary.to_string(index=False))
print()
print("uniform random MAP@3:", round((1 + 1/2 + 1/3) / 5, 4))

### Notes

Models 1 and 2 are small and educational. Qwen is the model that performs.

## 7. Submission

In [ ]:
if RUN_QWEN and qwen_results is not None:
    test_scores = score_many_questions(test_df)
    source = "Qwen2.5-32B"
else:
    test_scores = ranker_scores(build_features(test_df))
    source = "logistic regression"

top_three = np.argsort(-test_scores, axis=1)[:, :3]

predictions = []
for row in top_three:
    letters = []
    for option_number in row:
        letters.append(OPTIONS[option_number])
    predictions.append(" ".join(letters))

submission = pd.DataFrame({
    sample_submission.columns[0]: test_df["id"],
    sample_submission.columns[1]: predictions,
})

everything_valid = True
for prediction in submission[sample_submission.columns[1]]:
    letters = prediction.split()
    if len(letters) != 3:
        everything_valid = False
    if len(set(letters)) != 3:
        everything_valid = False
    for letter in letters:
        if letter not in OPTIONS:
            everything_valid = False

print("predictions from:", source)
print("rows            :", len(submission))
print("duplicate ids   :", submission[sample_submission.columns[0]].duplicated().sum())
print("all rows valid  :", everything_valid)

submission.to_csv("submission.csv", index=False)
print("saved submission.csv")

submission.head()

## 8. Conclusion

Simple baseline → classical ML → scratch neural attention → pretrained LLM.

Models 1 and 2 are intentionally small, to show the ideas working. Qwen is the
high-performance pretrained model and the one that clears the cutoff.

Two findings from the data:

- The dataset has many duplicate questions, so a grouped split is required.
  A random split inflates validation by about 0.3 MAP@3.
- Sorting options by length already beats random guessing, because the wrong
  options were made by rewriting the correct one.

Future work: fine-tune a mid-size encoder, add retrieval, remove the length
shortcut, use more orderings for the language model.